In [1]:
from google.colab import files
uploaded = files.upload()

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv


In [58]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [27]:
df = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [29]:
#data cleaning
#label encoding
cols = ['OnlineSecurity', 'MultipleLines', 'InternetService', 'Contract',
        'PaymentMethod','gender', 'Partner', 'Dependents', 'PhoneService', 'DeviceProtection',
         'TechSupport','StreamingTV', 'StreamingMovies', 'Dependents', 'PaperlessBilling', 'Churn',
        'OnlineBackup'
        ]
for col in cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
#data fill
cols = [
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for i in cols:
    mode_values = df[i].mode()

    if not mode_values.empty:
        df[i] = df[i].fillna(mode_values.iloc[0])
    else:
        print(f"{i} contains only null values")


In [34]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print(df["TotalCharges"].dtype)

float64


In [35]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,0,0,1,0,1,0,1,0,0,...,0,0,0,0,0,1,2,29.85,29.85,0
1,5575-GNVDE,1,0,0,0,34,1,0,0,2,...,2,0,0,0,1,0,3,56.95,1889.50,0
2,3668-QPYBK,1,0,0,0,2,1,0,0,2,...,0,0,0,0,0,1,3,53.85,108.15,1
3,7795-CFOCW,1,0,0,0,45,0,1,0,2,...,2,2,0,0,1,0,0,42.30,1840.75,0
4,9237-HQITU,0,0,0,0,2,1,0,1,0,...,0,0,0,0,0,1,2,70.70,151.65,1


In [36]:
x = df.drop(['Churn','customerID'], axis = 1)
y = df['Churn']

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(X_test)

In [38]:
X_train_tensor = torch.tensor(x_train_scaled, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype = torch.float32).view(-1,1)
X_test_tensor = torch.tensor(x_test_scaled, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype = torch.float32).view(-1,1)

In [41]:
# Create Tensor datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True
)

test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False
)

In [ ]:
#model initia lize

In [52]:
class ANN(nn.Module):
  def __init__(self):
    super(ANN, self).__init__()
    self.model = nn.Sequential(
        #1st hidden layer
        nn.Linear(X_train.shape[1],32),
        nn.ReLU(),
        #2nd hidden layer
        nn.Linear(32,32),
        nn.ReLU(),
        #output layer
        nn.Linear(32,1)
    )
  def forward (self,x):
    return self.model(x)

In [53]:
model = ANN()
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),lr=0.001,weight_decay=1e-4
)

In [54]:
#Train the model
epochs = 100
train_losses = []
val_losses = []
for epoch in range(epochs):
  model.train()
  running_loss =0.0
  for xb,yb in train_loader:
    optimizer.zero_grad()
    output = model(xb)
    loss = criterion(output,yb)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()
  epoch_train_loss = running_loss / len(train_loader)
  train_losses.append(epoch_train_loss)
  #validation state
  model.eval()
  running_val_loss =0.0
  with torch.no_grad():
     for xb,yb in test_loader:
          output = model(xb)
          loss = criterion(output,yb)
          running_val_loss += loss.item()
     epoch_val_loss = running_val_loss / len(test_loader)
     val_losses.append(epoch_val_loss)
     print(f"epoch{epoch+1}/{epochs}=> train loss = {epoch_train_loss} & vall loss = {epoch_val_loss}")

epoch1/100=> train loss = 0.49680352143648654 & vall loss = 0.42791912025875517
epoch2/100=> train loss = 0.4302477202678131 & vall loss = 0.418315894736184
epoch3/100=> train loss = 0.42444059362182507 & vall loss = 0.41303789085812037
epoch4/100=> train loss = 0.4219659610656695 & vall loss = 0.4112573352124956
epoch5/100=> train loss = 0.4192009649034274 & vall loss = 0.407205628686481
epoch6/100=> train loss = 0.41557385827188437 & vall loss = 0.40482860505580903
epoch7/100=> train loss = 0.4160088245141304 & vall loss = 0.4029580016930898
epoch8/100=> train loss = 0.41090173320581685 & vall loss = 0.4052595933278402
epoch9/100=> train loss = 0.41165243510135824 & vall loss = 0.40542309284210204
epoch10/100=> train loss = 0.40695375187248833 & vall loss = 0.40403598215844894
epoch11/100=> train loss = 0.4056056914693218 & vall loss = 0.40497801237636144
epoch12/100=> train loss = 0.40560566974898515 & vall loss = 0.40512024694018894
epoch13/100=> train loss = 0.4048359765339706 & v

In [55]:
# save model parameter
best_val_loss = float("inf")
if epoch_val_loss < best_val_loss:
  best_val_loss = epoch_val_loss
  torch.save(model.state_dict(),"best_model.pt")

In [56]:
torch.load("best_model.pt")
model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

In [59]:
# evaluatyion step
model.eval()
with torch.no_grad():
    train_probs = torch.sigmoid(model(X_train_tensor))
    test_probs = torch.sigmoid(model(X_test_tensor))

train_pred = (train_probs >= 0.5).int().numpy()
test_pred = (test_probs >= 0.5).int().numpy()

print("Train accuracy:", accuracy_score(y_train, train_pred))
print("Test accuracy:", accuracy_score(y_test, test_pred))
print(confusion_matrix(y_test, test_pred))
print(classification_report(y_test, test_pred))

Train accuracy: 0.8500177493787717
Test accuracy: 0.7920511000709723
[[920 116]
 [177 196]]
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1036
           1       0.63      0.53      0.57       373

    accuracy                           0.79      1409
   macro avg       0.73      0.71      0.72      1409
weighted avg       0.78      0.79      0.79      1409

